# Day 2 Lab: The Unscented Kalman Filter (UKF)
**Student Edition**

---
### 🎯 Learning Objectives
1. Understand the philosophy of the **Unscented Transform (UT)** vs. Taylor series linearization.
2. Master deterministic **Sigma Points** generation using Cholesky factorizations $(\mathbf{L}\mathbf{L}^T = \mathbf{P})$.
3. Implement the complete derivative-free **Unscented Kalman Filter** architecture.
4. Understand the physical rationale behind process and measurement covariance tuning in nonlinear filtering.
5. Correctly perform weighted mean and covariance recombination with circular angle wrapping.
6. Benchmark **UKF vs. EKF** on highly nonlinear trajectories with severe initial uncertainty via interactive Plotly charts.


## 1. Environment Setup


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2

np.random.seed(42)
print('Environment ready: NumPy, SciPy, and Plotly loaded.')


---
## 2. Theory: The Scaled Unscented Transform

For an $L$-dimensional state $\mathbf{x} \sim \mathcal{N}(\boldsymbol{\mu}, \mathbf{\Sigma})$:

$$\lambda = \alpha^2(L + \kappa) - L, \quad \gamma = \sqrt{L + \lambda}$$

$$\begin{aligned}
W_m^{(0)} &= \frac{\lambda}{L + \lambda}, \quad &W_c^{(0)} &= \frac{\lambda}{L + \lambda} + (1 - \alpha^2 + \beta) \\[4pt]
W_m^{(i)} &= \frac{1}{2(L + \lambda)}, \quad &W_c^{(i)} &= \frac{1}{2(L + \lambda)} \quad (i = 1, \dots, 2L)
\end{aligned}$$

Sigma Points via Cholesky decomposition $\mathbf{L}\mathbf{L}^T = \mathbf{\Sigma}$:
$$\boldsymbol{\mathcal{X}}^{(0)} = \boldsymbol{\mu}, \quad \boldsymbol{\mathcal{X}}^{(i)} = \boldsymbol{\mu} + \gamma \operatorname{col}_i(\mathbf{L}), \quad \boldsymbol{\mathcal{X}}^{(i+L)} = \boldsymbol{\mu} - \gamma \operatorname{col}_i(\mathbf{L})$$


---
## 3. Implementing the `UnscentedKalmanFilter` Class

### 📝 Exercise 1: Implement the UKF Class


In [ ]:
def wrap_angle(angle):
    """Wraps angle to [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi

class UnscentedKalmanFilter:
    """Unscented Kalman Filter (UKF) with Scaled Unscented Transform."""
    
    def __init__(self, x0: np.ndarray, P0: np.ndarray, alpha=0.5, beta=2.0, kappa=0.0):
        self.x = np.asarray(x0, dtype=np.float64).reshape(-1, 1)
        self.P = np.asarray(P0, dtype=np.float64)
        self.L = self.x.shape[0]
        self.alpha, self.beta, self.kappa = alpha, beta, kappa
        self.lambda_ = self.alpha**2 * (self.L + self.kappa) - self.L
        self.gamma = np.sqrt(self.L + self.lambda_)
        self.num_sigmas = 2 * self.L + 1
        self.Wm = np.zeros(self.num_sigmas)
        self.Wc = np.zeros(self.num_sigmas)
        self.Wm[0] = self.lambda_ / (self.L + self.lambda_)
        self.Wc[0] = self.Wm[0] + (1.0 - self.alpha**2 + self.beta)
        for i in range(1, self.num_sigmas):
            self.Wm[i] = 1.0 / (2.0 * (self.L + self.lambda_))
            self.Wc[i] = self.Wm[i]
            
    def generate_sigma_points(self, x_mean: np.ndarray, P_cov: np.ndarray):
        # TODO 1.1: Implement Cholesky factorization and 2L+1 sigma points generation
        pass
        
    def predict(self, f_func, Q: np.ndarray, u: np.ndarray = None, angle_indices: list = None):
        # TODO 1.2: Implement UKF prediction step
        pass
        
    def update(self, y: np.ndarray, h_func, R: np.ndarray, 
               meas_angle_indices: list = None, state_angle_indices: list = None):
        # TODO 1.3: Implement UKF measurement update
        pass


---
## 4. Benchmark Noise Formulations ($\mathbf{Q}$ and $\mathbf{R}$)

To rigorously test the Unscented Transform against 1st-order EKF linearization:
* **$\mathbf{Q} = \operatorname{diag}(0.05^2, 0.05^2, 0.1^2, 0.03^2)$:** Heading uncertainty $\sigma_\theta = 1.72^\circ$ reflects aggressive steering turn rates.
* **$\mathbf{R} = \operatorname{diag}(0.8^2, (\operatorname{deg2rad}(3.0))^2)$:** Large bearing noise $\sigma_\phi = 3.0^\circ$ is intentionally injected to test severe polar-to-Cartesian curvature.


---
## 5. Benchmark Challenge: EKF vs. UKF on Aggressive Nonlinear Maneuver


In [ ]:
dt = 0.1
T_total = 40.0
N_steps = int(T_total / dt)
time = np.linspace(0, T_total, N_steps)

x_true = np.array([10.0, 5.0, 12.0, 0.0]).reshape(4, 1)
x_true_all = np.zeros((4, N_steps))

Q = np.diag([0.05**2, 0.05**2, 0.1**2, 0.03**2])
R = np.diag([0.8**2, np.deg2rad(3.0)**2])

def motion_model(x, u):
    px, py, v, theta = x.flatten()
    a, omega = u.flatten()
    px_next = px + v * np.cos(theta) * dt
    py_next = py + v * np.sin(theta) * dt
    v_next = v + a * dt
    theta_next = theta + omega * dt
    return np.array([px_next, py_next, v_next, theta_next]).reshape(4, 1)

def measurement_model(x):
    px, py = x[0, 0], x[1, 0]
    r = np.sqrt(px**2 + py**2)
    phi = np.arctan2(py, px)
    return np.array([r, phi]).reshape(2, 1)

def get_F_jacobian(x):
    px, py, v, theta = x.flatten()
    return np.array([
        [1.0, 0.0, np.cos(theta) * dt, -v * np.sin(theta) * dt],
        [0.0, 1.0, np.sin(theta) * dt,  v * np.cos(theta) * dt],
        [0.0, 0.0, 1.0,                 0.0],
        [0.0, 0.0, 0.0,                 1.0]
    ])

def get_H_jacobian(x):
    px, py = x[0, 0], x[1, 0]
    r2 = max(px**2 + py**2, 1e-6)
    r = np.sqrt(r2)
    return np.array([
        [px / r,       py / r,       0.0, 0.0],
        [-py / r2,     px / r2,      0.0, 0.0]
    ])

measurements = []
for k in range(N_steps):
    t = time[k]
    a = 0.5 * np.cos(0.2 * t)
    omega = 0.25 * np.sin(0.15 * t)
    u = np.array([a, omega]).reshape(2, 1)
    w = np.random.multivariate_normal(np.zeros(4), Q).reshape(4, 1)
    x_true = motion_model(x_true, u) + w
    x_true[3, 0] = wrap_angle(x_true[3, 0])
    x_true_all[:, k] = x_true.flatten()
    v_noise = np.random.multivariate_normal(np.zeros(2), R).reshape(2, 1)
    y = measurement_model(x_true) + v_noise
    y[1, 0] = wrap_angle(y[1, 0])
    measurements.append((u, y))

x0_init = np.array([5.0, 1.0, 8.0, np.deg2rad(45.0)]).reshape(4, 1)
P0_init = np.diag([10.0**2, 10.0**2, 5.0**2, np.deg2rad(40.0)**2])

x_ekf = x0_init.copy()
P_ekf = P0_init.copy()
x_ekf_all = np.zeros((4, N_steps))

ukf = UnscentedKalmanFilter(x0_init, P0_init, alpha=0.5, beta=2.0, kappa=0.0)
x_ukf_all = np.zeros((4, N_steps))

for k in range(N_steps):
    u, y = measurements[k]
    # EKF
    F = get_F_jacobian(x_ekf)
    x_ekf = motion_model(x_ekf, u)
    P_ekf = F @ P_ekf @ F.T + Q
    H = get_H_jacobian(x_ekf)
    y_pred_ekf = measurement_model(x_ekf)
    res_ekf = y - y_pred_ekf
    res_ekf[1, 0] = wrap_angle(res_ekf[1, 0])
    S_ekf = H @ P_ekf @ H.T + R
    K_ekf = P_ekf @ H.T @ np.linalg.inv(S_ekf)
    x_ekf = x_ekf + K_ekf @ res_ekf
    x_ekf[3, 0] = wrap_angle(x_ekf[3, 0])
    P_ekf = (np.eye(4) - K_ekf @ H) @ P_ekf
    x_ekf_all[:, k] = x_ekf.flatten()

    # UKF
    ukf.predict(motion_model, Q, u=u, angle_indices=[3])
    ukf.update(y, measurement_model, R, meas_angle_indices=[1], state_angle_indices=[3])
    x_ukf_all[:, k] = ukf.x.flatten()

rmse_ekf_pos = np.sqrt(np.mean((x_true_all[0, :] - x_ekf_all[0, :])**2 + (x_true_all[1, :] - x_ekf_all[1, :])**2))
rmse_ukf_pos = np.sqrt(np.mean((x_true_all[0, :] - x_ukf_all[0, :])**2 + (x_true_all[1, :] - x_ukf_all[1, :])**2))

print(f'Position RMSE -> EKF: {rmse_ekf_pos:.3f} m | UKF: {rmse_ukf_pos:.3f} m')


---
## 6. Comparative Visualizations: UKF vs. EKF via Plotly


In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'Trajectory Tracking: EKF vs. UKF (EKF: {rmse_ekf_pos:.2f}m, UKF: {rmse_ukf_pos:.2f}m)',
        'Position Error Convergence Over Time'
    )
)

fig.add_trace(go.Scatter(x=x_true_all[0, :], y=x_true_all[1, :], mode='lines', name='Ground Truth', line=dict(color='black', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=x_ekf_all[0, :], y=x_ekf_all[1, :], mode='lines', name=f'EKF (RMSE: {rmse_ekf_pos:.2f}m)', line=dict(color='red', dash='dash', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=x_ukf_all[0, :], y=x_ukf_all[1, :], mode='lines', name=f'UKF (RMSE: {rmse_ukf_pos:.2f}m)', line=dict(color='blue', width=2)), row=1, col=1)

err_ekf = np.sqrt((x_true_all[0, :] - x_ekf_all[0, :])**2 + (x_true_all[1, :] - x_ekf_all[1, :])**2)
err_ukf = np.sqrt((x_true_all[0, :] - x_ukf_all[0, :])**2 + (x_true_all[1, :] - x_ukf_all[1, :])**2)
fig.add_trace(go.Scatter(x=time, y=err_ekf, mode='lines', name='EKF Error [m]', line=dict(color='red', dash='dash', width=1.5)), row=1, col=2)
fig.add_trace(go.Scatter(x=time, y=err_ukf, mode='lines', name='UKF Error [m]', line=dict(color='blue', width=1.5)), row=1, col=2)

fig.update_layout(title_text='Day 2 UKF vs. EKF Benchmark: Nonlinear Manifold Tracking', template='plotly_white', height=500, width=1100)
fig.show()


---
## 7. Summary
The UKF avoids Jacobians entirely, capturing true probability moments up to 3rd order with significantly greater resilience to severe non-linearities.
